<a href="https://colab.research.google.com/github/vedikamishra007/Big-Integer-cpp/blob/main/Final_Integration_Trial.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install gradio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.9/46.9 MB 20.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.2/322.2 kB 29.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.2/95.2 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 135.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.0/72.0 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.5/62.5 kB 6.5 MB/s eta 0:00:00


In [ ]:
import torch
from diffusers import DiffusionPipeline, DPMSolverMultistepScheduler
from diffusers.utils import export_to_video
import gradio as gr
import os

# Load the pipeline
pipe = DiffusionPipeline.from_pretrained("damo-vilab/text-to-video-ms-1.7b", torch_dtype=torch.float16, variant="fp16")
pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config)
pipe.enable_model_cpu_offload()

# Define the directory where videos will be saved
save_directory = "./generated_videos"
os.makedirs(save_directory, exist_ok=True)

# Function to generate video from text prompt
def generate_video(prompt):
    # Generate the frames from the text prompt
    video_frames = pipe(prompt, num_inference_steps=25).frames
    video_frames = video_frames.squeeze(0)  # Remove the leading dimension of size 1

    # Save the video to a temporary location
    video_path = export_to_video(video_frames)

    # Rename the video to .mp4 for compatibility
    mp4_path = video_path.replace(".gif", ".mp4")
    os.rename(video_path, mp4_path)

    # Save the video to the specified directory with a unique name
    saved_video_path = os.path.join(save_directory, f"output.mp4")
    os.rename(mp4_path, saved_video_path)

    # Clear the cache
    torch.cuda.empty_cache()

    return saved_video_path

# Create Gradio interface
gr.Interface(
    fn=generate_video,
    inputs=gr.Textbox(lines=2, placeholder="Enter a prompt, e.g., 'Doraemon is surfing'"),
    outputs=gr.Video(),
    title="Text-to-Video Generation",
    description="Please enter the text for the video you want to generate in below prompt bar and click 'Submit'",
).launch()


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model_index.json:   0%|          | 0.00/384 [00:00<?, ?B/s]

Fetching 12 files:   0%|          | 0/12 [00:00<?, ?it/s]

merges.txt:   0%|          | 0.00/525k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.06M [00:00<?, ?B/s]

...

diffusion_pytorch_model.fp16.safetensors:   0%|          | 0.00/167M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/657 [00:00<?, ?B/s]

diffusion_pytorch_model.fp16.safetensors:   0%|          | 0.00/2.82G [00:00<?, ?B/s]

Loading pipeline components...:   0%|          | 0/5 [00:00<?, ?it/s]

It looks like you are running Gradio on a hosted a Jupyter notebook. For the Gradio app to work, sharing must be enabled. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://70b1d19fe6aa703795.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
#Step 1) Extracting Frames from the generated video
import cv2
import os

# Function to extract frames from a video
def extract_frames(video_path, output_dir):
    # Create the output directory if it doesn't exist
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    # Capture the video
    cap = cv2.VideoCapture(video_path)

    # Initialize frame count
    frame_count = 0

    while True:
        # Read a frame from the video
        ret, frame = cap.read()

        # If no frame is returned, break the loop
        if not ret:
            break

        # Construct the filename for the frame
        frame_filename = os.path.join(output_dir, f'frame_{frame_count:04d}.jpg')

        # Save the frame as an image file
        cv2.imwrite(frame_filename, frame)

        # Increment frame count
        frame_count += 1

    # Release the video capture object
    cap.release()
    print(f'Total frames extracted: {frame_count}')

# Define the video path and output directory
video_path = '/content/generated_videos/output.mp4'  # Change this to your video path
output_dir = '/content/Input_Video_Frames'          # Output directory for frames

# Call the function to extract frames
extract_frames(video_path, output_dir)


Total frames extracted: 16


#Method 1) Unsharp Masking. By this method, we are taking each frame of video and enhancing it using Unsharp Masking Technique

In [ ]:
#Method 1) Unsharp Masking. By this method, we are taking each frame of video and enhancing it using Unsharp Masking Technique

import cv2
import os

def apply_filter_to_images(input_dir, output_dir):
    # Create the output directory if it doesn't exist
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    # Get list of image files in the input directory (jpg and png only)
    images = [img for img in os.listdir(input_dir) if img.endswith(".jpg") or img.endswith(".png")]

    if len(images) == 0:
        print("No images found in the input directory.")
        return

    # Loop through each image
    for img_name in images:
        img_path = os.path.join(input_dir, img_name)

        # Load the image
        image = cv2.imread(img_path)

        # Convert to grayscale
        gray_image = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

        # Apply Gaussian blur to the image
        blurred_image = cv2.GaussianBlur(gray_image, (5, 5), 0)

        # Create the unsharp mask
        sharpened_image = cv2.addWeighted(gray_image, 1.5, blurred_image, -0.5, 0)

        # Convert the sharpened image back to BGR format for saving
        enhanced_image = cv2.merge([sharpened_image, sharpened_image, sharpened_image])

        # Create the output image path
        output_img_path = os.path.join(output_dir, img_name)

        # Save the enhanced image
        cv2.imwrite(output_img_path, enhanced_image)

    print(f"Processed {len(images)} images and saved them to {output_dir}")

# Example usage:
input_dir = '/content/Input_Video_Frames'   # Provide the path to the folder with input images
output_dir = '/content/UnsharpMaskingEnhancedFrames'  # Provide the path to the folder to save enhanced images

# Apply the filter to all images in the input folder and save them in the output folder
apply_filter_to_images(input_dir, output_dir)


Processed 16 images and saved them to /content/UnsharpMaskingEnhancedFrames


In [ ]:
import cv2
import os

def create_video_from_frames(input_dir, output_video_path, frame_rate=5):
    # Get the list of image files in the input directory
    images = sorted([img for img in os.listdir(input_dir) if img.endswith(".jpg") or img.endswith(".png")])

    # Check if there are images in the input directory
    if len(images) == 0:
        print("No images found in the input directory.")
        return

    # Read the first image to get the dimensions (height, width)
    first_frame = cv2.imread(os.path.join(input_dir, images[0]))
    height, width, layers = first_frame.shape

    # Define the codec and create a VideoWriter object
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')  # Codec for .mp4
    out = cv2.VideoWriter(output_video_path, fourcc, frame_rate, (width, height))

    # Loop through all the images and write them to the video
    for image in images:
        img_path = os.path.join(input_dir, image)
        frame = cv2.imread(img_path)

        # Ensure the frame is in the correct dimensions (resize if necessary)
        if frame.shape[0] != height or frame.shape[1] != width:
            frame = cv2.resize(frame, (width, height))

        out.write(frame)

    # Release the VideoWriter object
    out.release()
    print(f"Video saved as {output_video_path}")

# Example usage:
input_dir = '/content/UnsharpMaskingEnhancedFrames'  # Replace with the folder containing your images
output_video_path = '/content/generated_videos/UnsharpMask_video.mp4'  # Replace with your desired output video path

# Call the function to create video from frames
create_video_from_frames(input_dir, output_video_path, frame_rate=5)


Video saved as /content/generated_videos/UnsharpMask_video.mp4


Method 2)Contrast Limited Adaptive Histogram Equalization(CLAHE Algorithm)

In [ ]:
import cv2
import os

def process_images(input_dir, output_dir):
    # Create output directory if it doesn't exist
    os.makedirs(output_dir, exist_ok=True)

    # Loop through all files in the input directory
    for filename in os.listdir(input_dir):
        # Construct full file path
        img_path = os.path.join(input_dir, filename)

        # Check if the file is an image
        if filename.lower().endswith(('.png', '.jpg', '.jpeg', '.tiff', '.bmp')):
            # Read the image
            img = cv2.imread(img_path)

            # Check if image is loaded successfully
            if img is None:
                print(f"Error loading image: {img_path}")
                continue

            # Convert the image to LAB color space
            lab = cv2.cvtColor(img, cv2.COLOR_BGR2Lab)

            # Split the LAB image into different channels
            l, a, b = cv2.split(lab)

            # Apply CLAHE to the L channel
            clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8))
            cl = clahe.apply(l)

            # Merge the CLAHE enhanced L channel with A and B channels
            limg = cv2.merge((cl, a, b))

            # Convert back to BGR color space
            img_clahe = cv2.cvtColor(limg, cv2.COLOR_Lab2BGR)

            # Construct output file path
            output_path = os.path.join(output_dir, filename)

            # Save the processed image
            cv2.imwrite(output_path, img_clahe)

            print(f"Processed and saved: {output_path}")

# Specify input and output directories
input_directory = "/content/Input_Video_Frames"
output_directory = "/content/CLAHE_Output_Frames"

# Call the function
process_images(input_directory, output_directory)


Processed and saved: /content/CLAHE_Output_Frames/frame_0012.jpg
Processed and saved: /content/CLAHE_Output_Frames/frame_0008.jpg
Processed and saved: /content/CLAHE_Output_Frames/frame_0009.jpg
Processed and saved: /content/CLAHE_Output_Frames/frame_0014.jpg
Processed and saved: /content/CLAHE_Output_Frames/frame_0011.jpg
Processed and saved: /content/CLAHE_Output_Frames/frame_0001.jpg
Processed and saved: /content/CLAHE_Output_Frames/frame_0000.jpg
Processed and saved: /content/CLAHE_Output_Frames/frame_0007.jpg
Processed and saved: /content/CLAHE_Output_Frames/frame_0015.jpg
Processed and saved: /content/CLAHE_Output_Frames/frame_0006.jpg
Processed and saved: /content/CLAHE_Output_Frames/frame_0013.jpg
Processed and saved: /content/CLAHE_Output_Frames/frame_0005.jpg
Processed and saved: /content/CLAHE_Output_Frames/frame_0004.jpg
Processed and saved: /content/CLAHE_Output_Frames/frame_0003.jpg
Processed and saved: /content/CLAHE_Output_Frames/frame_0010.jpg
Processed and saved: /con

In [ ]:
import cv2
import os

def create_video_from_frames(input_dir, output_video_path, frame_rate=5):
    # Get the list of image files in the input directory
    images = sorted([img for img in os.listdir(input_dir) if img.endswith(".jpg") or img.endswith(".png")])

    # Check if there are images in the input directory
    if len(images) == 0:
        print("No images found in the input directory.")
        return

    # Read the first image to get the dimensions (height, width)
    first_frame = cv2.imread(os.path.join(input_dir, images[0]))
    height, width, layers = first_frame.shape

    # Define the codec and create a VideoWriter object
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')  # Codec for .mp4
    out = cv2.VideoWriter(output_video_path, fourcc, frame_rate, (width, height))

    # Loop through all the images and write them to the video
    for image in images:
        img_path = os.path.join(input_dir, image)
        frame = cv2.imread(img_path)

        # Ensure the frame is in the correct dimensions (resize if necessary)
        if frame.shape[0] != height or frame.shape[1] != width:
            frame = cv2.resize(frame, (width, height))

        out.write(frame)

    # Release the VideoWriter object
    out.release()
    print(f"Video saved as {output_video_path}")

# Example usage:
input_dir = '/content/CLAHE_Output_Frames'  # Replace with the folder containing your images
output_video_path = '/content/generated_videos/CLAHE_video.mp4'  # Replace with your desired output video path

# Call the function to create video from frames
create_video_from_frames(input_dir, output_video_path, frame_rate=5)


Video saved as /content/generated_videos/CLAHE_video.mp4


Method 3) Median Filtering Technique

In [ ]:
import cv2
import os

def apply_median_filter(input_dir, output_dir, kernel_size=5):
    # Create output directory if it doesn't exist
    os.makedirs(output_dir, exist_ok=True)

    # Loop through all files in the input directory
    for filename in os.listdir(input_dir):
        # Construct the full path to the input image
        img_path = os.path.join(input_dir, filename)

        # Check if the file is an image
        if filename.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.tiff')):
            # Read the image
            img = cv2.imread(img_path)

            # Check if the image was loaded successfully
            if img is None:
                print(f"Error loading image: {img_path}")
                continue

            # Apply median filtering
            filtered_img = cv2.medianBlur(img, kernel_size)

            # Construct the full path to the output image
            output_path = os.path.join(output_dir, filename)

            # Save the filtered image
            cv2.imwrite(output_path, filtered_img)

            print(f"Processed and saved: {output_path}")

# Example usage
input_directory = '/content/Input_Video_Frames'  # Specify your input image directory here
output_directory = '/content/Median_Filtering_Enhanced_Frames'  # Specify your output directory here

# Apply median filtering and save the results
apply_median_filter(input_directory, output_directory, kernel_size=5)


Processed and saved: /content/Median_Filtering_Enhanced_Frames/frame_0012.jpg
Processed and saved: /content/Median_Filtering_Enhanced_Frames/frame_0008.jpg
Processed and saved: /content/Median_Filtering_Enhanced_Frames/frame_0009.jpg
Processed and saved: /content/Median_Filtering_Enhanced_Frames/frame_0014.jpg
Processed and saved: /content/Median_Filtering_Enhanced_Frames/frame_0011.jpg
Processed and saved: /content/Median_Filtering_Enhanced_Frames/frame_0001.jpg
Processed and saved: /content/Median_Filtering_Enhanced_Frames/frame_0000.jpg
Processed and saved: /content/Median_Filtering_Enhanced_Frames/frame_0007.jpg
Processed and saved: /content/Median_Filtering_Enhanced_Frames/frame_0015.jpg
Processed and saved: /content/Median_Filtering_Enhanced_Frames/frame_0006.jpg
Processed and saved: /content/Median_Filtering_Enhanced_Frames/frame_0013.jpg
Processed and saved: /content/Median_Filtering_Enhanced_Frames/frame_0005.jpg
Processed and saved: /content/Median_Filtering_Enhanced_Frames/f

In [ ]:
import cv2
import os

def create_video_from_frames(input_dir, output_video_path, frame_rate=5):
    # Get the list of image files in the input directory
    images = sorted([img for img in os.listdir(input_dir) if img.endswith(".jpg") or img.endswith(".png")])

    # Check if there are images in the input directory
    if len(images) == 0:
        print("No images found in the input directory.")
        return

    # Read the first image to get the dimensions (height, width)
    first_frame = cv2.imread(os.path.join(input_dir, images[0]))
    height, width, layers = first_frame.shape

    # Define the codec and create a VideoWriter object
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')  # Codec for .mp4
    out = cv2.VideoWriter(output_video_path, fourcc, frame_rate, (width, height))

    # Loop through all the images and write them to the video
    for image in images:
        img_path = os.path.join(input_dir, image)
        frame = cv2.imread(img_path)

        # Ensure the frame is in the correct dimensions (resize if necessary)
        if frame.shape[0] != height or frame.shape[1] != width:
            frame = cv2.resize(frame, (width, height))

        out.write(frame)

    # Release the VideoWriter object
    out.release()
    print(f"Video saved as {output_video_path}")

# Example usage:
input_dir = '/content/Median_Filtering_Enhanced_Frames'  # Replace with the folder containing your images
output_video_path = '/content/generated_videos/MedianFiltered_video.mp4'  # Replace with your desired output video path

# Call the function to create video from frames
create_video_from_frames(input_dir, output_video_path, frame_rate=5)


Video saved as /content/generated_videos/MedianFiltered_video.mp4


Method 4) Fast Non-Local Means Denoising (Colored)

In [ ]:
import cv2
import os

def process_and_save_images(input_dir, output_dir, h=10, hForColorComponents=10, templateWindowSize=7, searchWindowSize=21):
    # Create output directory if it doesn't exist
    os.makedirs(output_dir, exist_ok=True)

    # Loop through all files in the input directory
    for filename in os.listdir(input_dir):
        # Construct the full path to the input image
        img_path = os.path.join(input_dir, filename)

        # Check if the file is an image
        if filename.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.tiff')):
            # Read the image
            img = cv2.imread(img_path)

            # Check if the image was loaded successfully
            if img is None:
                print(f"Error loading image: {img_path}")
                continue

            # Apply Fast Non-Local Means Denoising (Colored)
            denoised_img = cv2.fastNlMeansDenoisingColored(img, None, h, hForColorComponents, templateWindowSize, searchWindowSize)

            # Construct the full path to the output image
            output_path = os.path.join(output_dir, filename)

            # Save the processed (denoised) image
            cv2.imwrite(output_path, denoised_img)

            print(f"Processed and saved: {output_path}")

# Example usage
input_directory = '/content/Input_Video_Frames'  # Specify your input image directory here
output_directory = '/content/FastNlEnhancedFrames'  # Specify your output directory here

# Apply denoising and save the results
process_and_save_images(input_directory, output_directory, h=10, hForColorComponents=10, templateWindowSize=7, searchWindowSize=21)


Processed and saved: /content/FastNlEnhancedFrames/frame_0012.jpg
Processed and saved: /content/FastNlEnhancedFrames/frame_0008.jpg
Processed and saved: /content/FastNlEnhancedFrames/frame_0009.jpg
Processed and saved: /content/FastNlEnhancedFrames/frame_0014.jpg
Processed and saved: /content/FastNlEnhancedFrames/frame_0011.jpg
Processed and saved: /content/FastNlEnhancedFrames/frame_0001.jpg
Processed and saved: /content/FastNlEnhancedFrames/frame_0000.jpg
Processed and saved: /content/FastNlEnhancedFrames/frame_0007.jpg
Processed and saved: /content/FastNlEnhancedFrames/frame_0015.jpg
Processed and saved: /content/FastNlEnhancedFrames/frame_0006.jpg
Processed and saved: /content/FastNlEnhancedFrames/frame_0013.jpg
Processed and saved: /content/FastNlEnhancedFrames/frame_0005.jpg
Processed and saved: /content/FastNlEnhancedFrames/frame_0004.jpg
Processed and saved: /content/FastNlEnhancedFrames/frame_0003.jpg
Processed and saved: /content/FastNlEnhancedFrames/frame_0010.jpg
Processed 

In [ ]:
import cv2
import os

def create_video_from_frames(input_dir, output_video_path, frame_rate=5):
    # Get the list of image files in the input directory
    images = sorted([img for img in os.listdir(input_dir) if img.endswith(".jpg") or img.endswith(".png")])

    # Check if there are images in the input directory
    if len(images) == 0:
        print("No images found in the input directory.")
        return

    # Read the first image to get the dimensions (height, width)
    first_frame = cv2.imread(os.path.join(input_dir, images[0]))
    height, width, layers = first_frame.shape

    # Define the codec and create a VideoWriter object
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')  # Codec for .mp4
    out = cv2.VideoWriter(output_video_path, fourcc, frame_rate, (width, height))

    # Loop through all the images and write them to the video
    for image in images:
        img_path = os.path.join(input_dir, image)
        frame = cv2.imread(img_path)

        # Ensure the frame is in the correct dimensions (resize if necessary)
        if frame.shape[0] != height or frame.shape[1] != width:
            frame = cv2.resize(frame, (width, height))

        out.write(frame)

    # Release the VideoWriter object
    out.release()
    print(f"Video saved as {output_video_path}")

# Example usage:
input_dir = '/content/FastNlEnhancedFrames'  # Replace with the folder containing your images
output_video_path = '/content/generated_videos/fastnl_output.mp4'  # Replace with your desired output video path

# Call the function to create video from frames
create_video_from_frames(input_dir, output_video_path, frame_rate=5)


Video saved as /content/generated_videos/fastnl_output.mp4


Final User Interface where user can compare 5 different videos

In [ ]:
import gradio as gr

def display_videos(video1, video2, video3, video4, video5):
    return video1, video2, video3, video4, video5

# Paths to the .mp4 files you want to display
video1_path = "/content/generated_videos/output.mp4"
video2_path = "/content/generated_videos/UnsharpMask_video.mp4"
video3_path = "/content/generated_videos/CLAHE_video.mp4"
video4_path = "/content/generated_videos/MedianFiltered_video.mp4"
video5_path = "/content/generated_videos/fastnl_output.mp4"

# Gradio interface for comparing five videos
with gr.Blocks() as demo:
    gr.Markdown("## This UI is to compare videos enhanced using different techniques")

    # Set custom dimensions for video display
    video_width = 400  # Set to a medium size, adjust as needed
    video_height = 300  # Set to a medium size, adjust as needed

    # Video components for each video, displayed one below the other
    video1 = gr.Video(label="Original Video", value=video1_path, width=video_width, height=video_height)
    video2 = gr.Video(label="Unsharp Masked Video", value=video2_path, width=video_width, height=video_height)
    video3 = gr.Video(label="CLAHE Enhanced Video", value=video3_path, width=video_width, height=video_height)
    video4 = gr.Video(label="Median Filter Enhanced Video", value=video4_path, width=video_width, height=video_height)
    video5 = gr.Video(label="Fast Non-Local Denoising Enhanced Video", value=video5_path, width=video_width, height=video_height)

demo.launch()


/usr/local/lib/python3.11/dist-packages/gradio/components/video.py:341: UserWarning: Video does not have browser-compatible container or codec. Converting to mp4.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/gradio/components/video.py:341: UserWarning: Video does not have browser-compatible container or codec. Converting to mp4.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/gradio/components/video.py:341: UserWarning: Video does not have browser-compatible container or codec. Converting to mp4.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/gradio/components/video.py:341: UserWarning: Video does not have browser-compatible container or codec. Converting to mp4.
  warnings.warn(


Running Gradio in a Colab notebook requires sharing enabled. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://9a0e08797705dda38b.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
